In [1]:
#import
import ConnectionConfig as cc
debugging_mode=True

In [2]:
#config

cc.setupEnvironment()
spark = cc.startLocalCluster("mongodbsetup",4)
spark.getActiveSession()

Environment variables are set...


In [3]:
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")

In [4]:
#EXTRACT

#treusure tabel
df_treasure = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_treasure.createOrReplaceTempView("treasure")

#treusure tabel
df_stage = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stage") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_stage.createOrReplaceTempView("stage")

#treusure tabel
df_treasure_stage = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure_stages") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_treasure_stage.createOrReplaceTempView("treasure_stages")

#city tabel
df_city = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "city") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_city.createOrReplaceTempView("city")

#country tabel
df_country = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "country") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_country.createOrReplaceTempView("country")


In [5]:
#TRANSFORM
treasure_mongoDb = spark.sql("""
                             SELECT t.id,
                                    t.difficulty,
                                    t.terrain,
                                    t.owner_id,
                                    co.name as country,
                                            named_struct(
                                                'id',c.city_id,
                                                    'name',c.city_name,
                                                    'latitude', c.latitude,
                                                    'longitude', c.longitude
                                            ) AS city,
                                    collect_list(
                                            named_struct(
                                                    'stage_id', st.id,
                                                    'container_size', st.container_size,
                                                    'description', st.description,
                                                    'latitude', st.latitude,
                                                    'longitude', st.longitude,
                                                    'sequence_number', st.sequence_number,
                                                    'type', st.type,
                                                    'visibility', st.visibility
                                            )
                                    ) AS stages
                             FROM treasure t
                                      JOIN city c ON t.city_city_id = c.city_id
                                      JOIN country co ON c.country_code = co.code
                                      LEFT JOIN treasure_stages ts ON t.id = ts.treasure_id
                                      LEFT JOIN stage st ON ts.stages_id = st.id
                             GROUP BY t.id, t.difficulty, t.terrain, t.owner_id,
                                      c.city_id, c.city_name, c.latitude, c.longitude, c.postal_code, c.country_code,
                                      co.code3, co.name
                             """)

treasure_mongoDb.createOrReplaceTempView("treasure_mongoDb")



In [6]:
#LOAD

treasure_mongoDb.coalesce(1).write \
    .format("json") \
    .mode("overwrite") \
    .save("treasures_export.json")